# Exercise A: Chain-of-Thought Prompting 

## Task (30 Minutes)
For this task you will be given a small subset of examples from the GSM8K and GSM8K-Platinum datasets. Your task will be to perform some prompt engineering:

1.1. Prompt the model to answer the question without reasoning.

1.2. Rewrite the prompt with a CoT-style instruction.

1.3. ~ to make it 1-shot CoT (Hint: take any problem and step-by-step solution to the prompt from here: https://huggingface.co/datasets/openai/gsm8k).

1.4. ~ to add a misleading CoT-style instruction.

1.5. ~ irrelevant CoT-style instruction.

And compute accuracies for every prompt type. 

Don't forget to set `max_new_tokens` to a higher value to allow the model to reason!

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
import json
import re
from collections import defaultdict
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
device = torch.device("cuda:0")  # Or select a different device
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype="auto")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")

In [ ]:
def convert_to_chat(prompt: str, tokenizer: AutoTokenizer, sys_prompt: str):
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": prompt}
        ],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

In [ ]:
# These may serve as a guideline on how to evaluate the outputs later. You needn't follow them, though.
def extract_final_answer(text):
    pass

def compute_accuracy_by_prompt(results):
    pass

In [ ]:
sys_prompt = "You are a competent math expert."

# Example prompt
prompt_simple = """You will be given a math problem. The solution to the problem is an integer. Your task is to provide the solution. 
Only provide the final answer as an integer, do not add any reasoning. But make sure that your final answer (the integer) starts with 'FINAL ANSWER:'.\n\n
The math problem is: {problem}"""

# Exercise B: Building an LLM Agent with Tools

In [ ]:
# ! pip install transformers torch accelerate requests

In [ ]:
# IMPORTS
import json
import math
import re
import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

We provide you with a working agent loop and a calculator tool. You do not need to touch these parts, but study them as a reference.

We will use `Qwen/Qwen2.5-1.5B-Instruct` as our model.


In [ ]:
# CONFIG

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 512
MAX_ITERATIONS = 8

## How We Define Tools

Every tool has two parts that must match each other. We will explain them based on the calculator tool which is already implemented. Later, you will add your own tools. First, study this as an example.


**Part A:  The Python function**
```
def my_tool(param: str) -> str:
    result = ...        # do the actual work
    return str(result)  # always return a string
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# REFERENCE TOOL — Calculator
# ─────────────────────────────────────────────────────────────────────────────
 
def calculator(expression: str) -> str:
    """Safely evaluate a math expression using the math module."""
    try:
        import math
        safe_env = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
        result = eval(expression, {"__builtins__": {}, "math": math}, safe_env)

        return str(result)
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"



**Part B: The JSON schema (tells the model what the tool does and what it expects)**
```
{
    "type": "function",
    "function": {
        "name": "my_tool",             # must match the function name
        "description": "...",          # the model reads this to decide when to use the tool
        "parameters": {
            "type": "object",
            "properties": {
                "param": {
                    "type": "string",
                    "description": "..."  # the model reads this to know what to pass
                }
            },
            "required": ["param"]
        }
    }
}
```


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": (
                "Evaluate a mathematical expression and return the result. "
                "Supports arithmetic (+, -, *, /, **) and math functions: "
                "sqrt(), sin(), cos(), log(), log10(), ceil(), floor(). "
                "Constants 'pi' and 'e' are available. "
                "Examples: '2 + 3 * 4', 'sqrt(144)', 'pi * 7**2'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "A valid Python math expression to compute.",
                    }
                },
                "required": ["expression"],
            },
        },
    },
 
    # ── TODO add the new tool schemas here ──────────────────────
    
]
 



Then we need to register the function like this:
```
TOOL_REGISTRY["my_tool"] = my_tool
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# TOOL REGISTRY
# (Add your tools here once you implement them)
# ─────────────────────────────────────────────────────────────────────────────
 
# Maps tool name → Python function
TOOL_REGISTRY = {
    "calculator": calculator,
    # TODO add your tools here
}

## Agent Infrastructure

In [ ]:
def load_model(model_id: str = MODEL_ID):
    print(f"Loading model: {model_id}  (first run downloads weights)\n")
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype, # TODO SETUP YOUR DEVICE IF NEEDED: device_map="auto", low_cpu_mem_usage=True
    )
    model.eval()
    print(f"Model loaded on: {next(model.parameters()).device}\n")
    return model, tokenizer
 
 
def generate(model, tokenizer, messages, tools=None):
    text = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True, tokenize=False
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
 
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
 
 
def parse_tool_calls(response: str):
    pattern = r"<tool_call>\s*(.*?)\s*</tool_call>" # parse the tool call by <tool_call> and </tool_call>
    matches = re.findall(pattern, response, re.DOTALL)
    tool_calls = []
    for raw in matches:
        try:
            tool_calls.append(json.loads(raw))
        except json.JSONDecodeError:
            pass
    return tool_calls
 
 
def run_agent(question: str, model, tokenizer):
    messages = [{"role": "user", "content": question}]
    print(f"\n{'='*60}\n  User: {question}\n{'='*60}")
 
    for _ in range(MAX_ITERATIONS):
        response = generate(model, tokenizer, messages, tools=TOOLS)
        tool_calls = parse_tool_calls(response)
 
        if not tool_calls:
            answer = re.sub(r"<[^>]+>", "", response).strip()
            print(f"\n🤖  Answer: {answer}")
            return answer
 
        messages.append({"role": "assistant", "content": response})
 
        for tc in tool_calls:
            fn_name = tc.get("name")
            fn_args = tc.get("arguments", {})
            if isinstance(fn_args, str):
                fn_args = json.loads(fn_args)
 
            print(f"\n🔨  Tool call  : {fn_name}({fn_args})")
 
            result = (
                TOOL_REGISTRY[fn_name](**fn_args)
                if fn_name in TOOL_REGISTRY
                else f"Unknown tool: '{fn_name}'"
            )
            print(f"  Tool result: {result}")
 
            messages.append({"role": "tool", "name": fn_name, "content": result})
 
    print("Max iterations reached.")
    return None

In [ ]:
model, tokenizer = load_model()
 

run_agent("What is the square root of 1764, minus 18?", model, tokenizer)

## Task 1: Unit Converter (10 minutes)

Implement a `unit_converter` tool that handles common conversions.

Your tool short support:

| From Unit | To Unit    | Formula                      |
|-----------|-------------|------------------------------|
| Celsius   | Fahrenheit  | (value × 9/5) + 32          |
| Fahrenheit| Celsius     | (value - 32) × 5/9          |
| KM        | Miles       | value × 0.621371            |
| Miles     | KM          | value × 1.60934             |
| KG        | Pounds      | value × 2.20462             |
| Pounds    | KG          | value × 0.453592            |

* Return a human-readable string, e.g. "42.0 km = 26.10 miles"
* Handle unknown units gracefully with an error message
* The model will pass value as a float. It may also pass it as a string, so cast with float(value) to be safe


In [ ]:
def unit_converter(value: float, from_unit: str, to_unit: str) -> str:
    raise NotImplementedError("Not implemented")

In [ ]:
# Test the agent with the following questions.

test_question_a = "I ran 10 miles this morning. How many kilometers is that?"
test_question_b = "Convert 37.5 degrees Celsius to Fahrenheit."

## Task 2: Weather Tool (25 minutes)

Implement a `get_weather` tool that fetches live weather for any city using the Open-Meteo API (free, no API key needed). You can find the documentation [here](https://open-meteo.com/).

The call requires two steps:
1. Get coordinates for a city name (geocoding search)
2. Get current weather using the coordinates (forecast)


* Use the `requests` library: `import requests`
* Always wrap API calls in try/except. The tool must return a string even on failure
* Return a readable summary, e.g.: `"Berlin, Germany: 18.3°C, wind 12.4 km/h, Partly cloudy"`



In [ ]:
# Partial WMO weather code lookup — feel free to extend this
WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Icy fog", 51: "Light drizzle", 53: "Drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
    80: "Slight showers", 81: "Moderate showers", 82: "Heavy showers",
    95: "Thunderstorm", 99: "Thunderstorm with hail",
}

In [ ]:
def get_weather(city: str) -> str:
    raise NotImplementedError("Not implemented")

# Test the agent with the following question.

test_question = "What's the weather in Bochum right now?"


## Task 3: Multi-Step Questions (5 minutes)

Now that all tools are registered, test your agent with questions that require chaining multiple tools. Observe the full Thought → Tool call → Observation output.

Try these:

1. "It's currently 95°F in Phoenix. What is that in Celsius, and what does weather typically feel like at that temperature?"
2. "What is the weather in Tokyo, and convert the temperature to Fahrenheit for me?"
3. (Make up your own!)

What to observe:

* Does the model decide on its own which tool to call?
* Does it call tools in the right order?
* What happens if you ask something neither tool can answer. Does it say so gracefully?

In [ ]:
# YOUR CODE HERE